# Chatbot Monitoring with the Galileo Python SDK

This notebook keeps the Galileo SDK calls inline so the setup and instrumentation are visible as executable code blocks. The goal is to show the actual sequence of SDK operations with minimal helper abstraction.


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


In [13]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.log_streams import create_log_stream, enable_metrics
from galileo.openai import openai
from galileo.projects import delete_project, get_project

PROJECT_NAME = 'GalileoEval_S1_Chatbot'
LOG_STREAM_NAME = 'chatbot-monitoring'
MODEL = 'gpt-4o-mini'

PROJECT_NAME, LOG_STREAM_NAME, MODEL


('GalileoEval_S1_Chatbot', 'chatbot-monitoring', 'gpt-4o-mini')

## 1. Initialize Galileo

Initialize the Galileo context, make sure the log stream exists, and print the console links if Galileo returns IDs.


## GalileoPythonConfig Overview

- **`GalileoPythonConfig`** extends `GalileoConfig` and provides configuration via:
  - Environment variables with prefix `GALILEO_` (e.g. from your `.env`)
  - Config file `galileo-python-config.json` in the Galileo home directory (default: `~/.galileo`)
  - Direct attributes set on the config object

### Main Attributes (from SDK source):

| Attribute              | Purpose                                                                 |
|------------------------|-------------------------------------------------------------------------|
| `console_url`          | Base URL of the Galileo web UI (used for notebook project links)         |
| `api_url`              | Galileo API base URL (custom/on-prem support)                           |
| `api_key`              | Your `GALILEO_API_KEY` for authentication                               |
| `username` / `password`| Optional login credentials                                              |
| `home_dir`             | Galileo config directory (default `~/.galileo`)                         |
| `log_level`            | SDK log level                                                           |
| `validated_api_client` | API client after successful auth (internal use)                         |

### How it's Used in This Notebook

This notebook mainly uses `config.console_url` to construct Galileo Console URLs:

```python
project_url = f'{config.console_url}project/{project_id}'
log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
```

In [14]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

try:
    create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')


https://console.demo-v2.galileocloud.io/project/979c1e2b-bbeb-4ed1-9977-9dabc0513fda
https://console.demo-v2.galileocloud.io/project/979c1e2b-bbeb-4ed1-9977-9dabc0513fda/log-streams/6daf77cf-641f-4fae-ae0e-bfcb7718a355


## 2. Create an OpenAI Client Through Galileo

The Galileo OpenAI wrapper instruments the calls automatically, so the SDK usage stays close to normal OpenAI client code.


In [15]:
client = openai.OpenAI()
client


## 3. Log a Single Chat Completion

The simplest pattern is: initialize context, make a model call, flush the Galileo context.


In [18]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful customer support agent for a software company.'},
        {'role': 'user', 'content': 'How do I reset my password?'},
    ],
    max_tokens=100,
)

galileo_context.flush()
response.choices[0].message.content


'To reset your password, please follow these steps:\n\n1. **Go to the Login Page**: Visit the login page of the application or website.\n\n2. **Click on \'Forgot Password?\'**: Look for a link or button that says "Forgot Password?" or "Reset Password" and click on it.\n\n3. **Enter Your Email Address**: You will usually be prompted to enter the email address associated with your account.\n\n4. **Check Your Email**: After submitting your email,'

## 4. Log a Streaming Response

**Why chunks?** With `stream=True`, the API doesn't return one final message. It yields **chunks** as the model generates tokens—each chunk has a small piece of the reply (a **delta**). You loop over the stream and collect `chunk.choices[0].delta.content`, then join the chunks to get the full response.

**Why flush after?** Galileo logs a complete trace (input + full assistant reply). While streaming, the reply isn't complete until you've consumed all chunks. So you accumulate the deltas, then call `galileo_context.flush()` once the stream is done. That way Galileo receives the full response and can log the trace correctly.


In [21]:
project = get_project(name=PROJECT_NAME)
if project:
    delete_project(name=PROJECT_NAME)

galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful customer support agent.'},
        {'role': 'user', 'content': 'What are your business hours?'},
    ],
    max_tokens=80,
    stream=True,
)

chunks = []
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chunks.append(chunk.choices[0].delta.content)

galileo_context.flush()
''.join(chunks)


'Our business hours may vary based on the specific service or location you are inquiring about. Generally, most customer support services operate Monday through Friday from 9 AM to 5 PM, but some may have extended hours or weekend support. For accurate hours, please check our website or contact us directly. How can I assist you further?'

## 5. Track a Multi-Turn Session

Use `start_session()` before a sequence of related turns to group the conversation in Galileo.

**Visual: one session, three LLM turns**

```
start_session('customer-session-001')
        │
        ▼
┌─────────────────────────────────────────────────────────────────┐
│  Turn 1                                                         │
│  User: "I can't log into my account."                           │
│    → LLM (first)  →  conversation updated                       │
├─────────────────────────────────────────────────────────────────┤
│  Turn 2                                                         │
│  User: "I tried that but it still doesn't work."                 │
│    → LLM (second)  →  conversation updated                      │
├─────────────────────────────────────────────────────────────────┤
│  Turn 3                                                         │
│  User: "Okay that worked, thank you!"                           │
│    → LLM (final_response)                                       │
└─────────────────────────────────────────────────────────────────┘
        │
        ▼
galileo_context.flush()   ←  send one trace for the whole session
```

**Why flush once after all turns?** A *session* in Galileo is one logical conversation. Flushing after every turn would create separate traces (one per turn) instead of a single trace that contains the full multi-turn exchange. Flushing at the end sends one trace with the complete conversation, so you can inspect the whole thread and all model calls together in the Galileo console.


In [ ]:
project = get_project(name=PROJECT_NAME)
if project:
    delete_project(name=PROJECT_NAME)

galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
galileo_context.start_session(name='customer-session-001')

conversation = [
    {'role': 'system', 'content': 'You are a customer support agent. Be concise.'},
    {'role': 'user', 'content': "I can't log into my account."},
]

first = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)
conversation.append({'role': 'assistant', 'content': first.choices[0].message.content})
conversation.append({'role': 'user', 'content': "I tried that but it still doesn't work."})

second = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)
conversation.append({'role': 'assistant', 'content': second.choices[0].message.content})
conversation.append({'role': 'user', 'content': 'Okay that worked, thank you!'})

final_response = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=80)

galileo_context.flush()
final_response.choices[0].message.content


"You're welcome! I'm glad it worked. If you need any more assistance, feel free to ask!"

## 6. Enable Quality and Safety Metrics

Metrics are configured on the log stream, then Galileo scores future traces for those dimensions.


In [22]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.input_pii,
        GalileoMetrics.output_pii,
        GalileoMetrics.input_toxicity,
        GalileoMetrics.output_toxicity,
    ],
)
project_url = f'{config.console_url}project/{project_id}'
log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
print(project_url)
print(log_stream_url)

https://console.demo-v2.galileocloud.io/project/979c1e2b-bbeb-4ed1-9977-9dabc0513fda
https://console.demo-v2.galileocloud.io/project/979c1e2b-bbeb-4ed1-9977-9dabc0513fda/log-streams/6daf77cf-641f-4fae-ae0e-bfcb7718a355


### check the metrics in the console here ^


## 7. Clean Up the Demo Project

Run this only when you want to remove the Galileo project created by the notebook.


In [ ]:
delete_project(name=PROJECT_NAME)
